In [1]:
import sqlite3
import pandas as pd
import os

In [2]:
LEGACY_DB  = "data/legacy.db"
EXPORT_DIR = "data/exports"
os.makedirs(EXPORT_DIR, exist_ok=True)
 
src = sqlite3.connect(LEGACY_DB)
dst = sqlite3.connect("data/migrated.db")

In [3]:
tables = ["customers", "accounts", "billing_cycles", "payments"]
kpi_rows = []
 
for t in tables:
    src_n = pd.read_sql(f"SELECT COUNT(*) AS n FROM {t}", src)["n"][0]
    dst_n = pd.read_sql(f"SELECT COUNT(*) AS n FROM {t}", dst)["n"][0]
    kpi_rows.append({
        "table":          t,
        "source_count":   src_n,
        "migrated_count": dst_n,
        "dropped":        src_n - dst_n,
        "migration_rate_pct": round(dst_n / src_n * 100, 2) if src_n > 0 else 0
    })
 
# Total billing discrepancy €
src_cyc = pd.read_sql("SELECT cycle_id, amount_due FROM billing_cycles", src)
dst_cyc = pd.read_sql("SELECT cycle_id, amount_due FROM billing_cycles", dst)
merged  = src_cyc.merge(dst_cyc, on="cycle_id", suffixes=("_src", "_dst"))
merged["disc"] = (merged["amount_due_src"] - merged["amount_due_dst"]).abs()
total_disc_eur = round(merged["disc"].sum(), 2)
 
pd.DataFrame(kpi_rows).to_csv(f"{EXPORT_DIR}/pbi_summary_cards.csv", index=False)
print(f"✅ pbi_summary_cards.csv")
print(f"   Total billing discrepancy: €{total_disc_eur:,.2f}")

✅ pbi_summary_cards.csv
   Total billing discrepancy: €37,741.48


In [4]:
q = """
    SELECT
        c.region,
        COUNT(bc.cycle_id)          AS total_cycles,
        SUM(bc.amount_due)          AS total_billed,
        COALESCE(SUM(p.amount_paid),0) AS total_paid
    FROM customers c
    JOIN accounts a        ON c.customer_id = a.customer_id
    JOIN billing_cycles bc ON a.account_id  = bc.account_id
    LEFT JOIN payments p   ON bc.cycle_id   = p.cycle_id
    GROUP BY c.region
    ORDER BY total_billed DESC
"""
df_region = pd.read_sql(q, src)
df_region["outstanding"] = (df_region["total_billed"] - df_region["total_paid"]).round(2)
df_region.to_csv(f"{EXPORT_DIR}/pbi_discrepancies_by_region.csv", index=False)
print(f"✅ pbi_discrepancies_by_region.csv")

✅ pbi_discrepancies_by_region.csv


In [5]:
q2 = """
    SELECT
        c.fuel_type,
        COUNT(bc.cycle_id)          AS total_cycles,
        SUM(bc.amount_due)          AS total_billed,
        COALESCE(SUM(p.amount_paid),0) AS total_paid,
        SUM(CASE WHEN bc.status = 'Overdue' THEN 1 ELSE 0 END) AS overdue_cycles
    FROM customers c
    JOIN accounts a        ON c.customer_id = a.customer_id
    JOIN billing_cycles bc ON a.account_id  = bc.account_id
    LEFT JOIN payments p   ON bc.cycle_id   = p.cycle_id
    GROUP BY c.fuel_type
"""
df_fuel = pd.read_sql(q2, src)
df_fuel["outstanding"] = (df_fuel["total_billed"] - df_fuel["total_paid"]).round(2)
df_fuel.to_csv(f"{EXPORT_DIR}/pbi_discrepancies_by_fuel.csv", index=False)
print(f"✅ pbi_discrepancies_by_fuel.csv")

✅ pbi_discrepancies_by_fuel.csv


In [6]:
df_bc = pd.read_sql("SELECT cycle_start, amount_due, status FROM billing_cycles", src)
 
def extract_year_month(date_str):
    try:
        from datetime import datetime
        d = datetime.strptime(date_str.strip(), "%d/%m/%Y")
        return d.strftime("%Y-%m")
    except:
        return None
 
df_bc["year_month"] = df_bc["cycle_start"].apply(extract_year_month)
df_trend = (
    df_bc.dropna(subset=["year_month"])
    .groupby("year_month")
    .agg(
        total_cycles=("amount_due", "count"),
        total_billed=("amount_due", "sum"),
        overdue_cycles=("status", lambda x: (x == "Overdue").sum())
    )
    .reset_index()
    .sort_values("year_month")
)
df_trend["total_billed"] = df_trend["total_billed"].round(2)
df_trend.to_csv(f"{EXPORT_DIR}/pbi_billing_trend.csv", index=False)
print(f"✅ pbi_billing_trend.csv")

✅ pbi_billing_trend.csv


In [8]:
master = pd.read_csv(f"{EXPORT_DIR}/recon_master.csv")
 
cust_info = pd.read_sql("""
    SELECT a.account_id, c.region, c.fuel_type, c.tariff_plan, c.is_active
    FROM accounts a
    JOIN customers c ON a.customer_id = c.customer_id
""", src)
 
master_enriched = master.merge(cust_info, on="account_id", how="left")
master_enriched.to_csv(f"{EXPORT_DIR}/pbi_recon_master.csv", index=False)
print(f"✅ pbi_recon_master.csv ({len(master_enriched):,} rows)")
 
src.close()
dst.close()

ProgrammingError: Cannot operate on a closed database.